In [4]:
import librosa
import numpy as np
from pydub import AudioSegment


c:\Users\jiyon\Desktop\Programming_projects\UNSW\musictransformationanalysis\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [5]:
import librosa
import numpy as np

# Paths to vocal tracks
vocal_paths = {
    "Kanye West": "./song1/vocals.mp3",
    "Cardi B": "./song2/vocals.mp3",
    "Elton John": "./song3/vocals.mp3",
    "Tom Macdonald": "./song4/vocals.mp3"
}

# Load all vocal tracks (trim to same duration for fair comparison)
max_duration = 180  # Compare first 3 minutes
vocals = {}
for name, path in vocal_paths.items():
    y, sr = librosa.load(path, duration=max_duration, mono=True)
    vocals[name] = {"y": y, "sr": sr}

In [6]:
def extract_pitch(y, sr):
    pitch = librosa.yin(y, fmin=50, fmax=2000)  # Robust pitch tracking
    pitch = pitch[pitch > 0]  # Remove unvoiced segments
    return {
        "mean_pitch": np.mean(pitch),
        "pitch_std": np.std(pitch),
        "pitch_range": np.max(pitch) - np.min(pitch)
    }

def extract_intensity(y, sr):
    S = np.abs(librosa.stft(y))
    intensity_db = librosa.amplitude_to_db(S, ref=np.max)
    return np.mean(intensity_db)

# Analyze all tracks
for name, data in vocals.items():
    y, sr = data["y"], data["sr"]
    data.update(extract_pitch(y, sr))
    data["intensity"] = extract_intensity(y, sr)

In [ ]:
import parselmouth

def extract_formants(y, sr):
    sound = parselmouth.Sound(y, sampling_frequency=sr)
    formants = sound.to_formant_burg()
    f1 = formants.get_value_at_time(1, 0.5, 'HERTZ', 'NEAREST')  # First formant
    f2 = formants.get_value_at_time(2, 0.5, 'HERTZ', 'NEAREST')  # Second formant
    return {"f1": f1, "f2": f2}

for name, data in vocals.items():
    data.update(extract_formants(data["y"], data["sr"]))